# Generate Evaluation Dataset with Snowflake Cortex

Produces 10 simple (question, ADQL) pairs per intent that match the structure of the queries built by the pipeline.

In [ ]:
!pip install snowflake-connector-python python-dotenv -q

In [1]:
import snowflake.connector
from dotenv import load_dotenv
import os
import json
import re
import pandas as pd

def get_snowflake_connection():
    load_dotenv()
    totp_code = input('Enter your TOTP code (6 digits): ')
    conn = snowflake.connector.connect(
        user=os.getenv('SNOWFLAKE_USER'),
        password=os.getenv('SNOWFLAKE_PASSWORD'),
        account=os.getenv('SNOWFLAKE_ACCOUNT'),
        passcode=totp_code,
    )
    cursor = conn.cursor()
    cursor.execute('USE WAREHOUSE CHIPMUNK')
    print('Snowflake connected, warehouse CHIPMUNK in use')
    return conn, cursor

conn, cursor = get_snowflake_connection()

Snowflake connected, warehouse CHIPMUNK in use


In [2]:
# ── pipeline contract ────────────────────────────────────────────────────────
VALID_COLUMNS = [
    'ra', 'dec', 'source_id', 'parallax', 'parallax_error',
    'pmra', 'pmdec', 'radial_velocity',
    'phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag', 'bp_rp',
    'teff_gspphot', 'logg_gspphot', 'mh_gspphot',
    'phot_variable_flag', 'ruwe',
]

# Only the 4 intents that admit truly simple, single-filter queries.
INTENTS = ['cone_search', 'stellar_population', 'nearest_neighbours', 'hr_diagram']

INTENT_TEMPLATES = {
    'cone_search': (
        "stars in a small sky region around a named target. "
        "Template: SELECT TOP <N> source_id, ra, dec, phot_g_mean_mag, bp_rp, pmra, pmdec "
        "FROM gaiadr3.gaia_source "
        "WHERE 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', <ra>, <dec>, <radius>)) "
        "ORDER BY random_index"
    ),
    'stellar_population': (
        "stars filtered by one astrophysical property, sky-wide. "
        "Template: SELECT TOP <N> source_id, ra, dec, <one_property>, phot_g_mean_mag, bp_rp, pmra, pmdec "
        "FROM gaiadr3.gaia_source "
        "WHERE <one_property> IS NOT NULL AND <one_simple_filter> "
        "ORDER BY random_index"
    ),
    'nearest_neighbours': (
        "the closest stars to the Sun. "
        "Template: SELECT TOP <N> parallax, source_id, ra, dec, phot_g_mean_mag, bp_rp, pmra, pmdec "
        "FROM gaiadr3.gaia_source "
        "WHERE parallax > 0 "
        "ORDER BY parallax DESC"
    ),
    'hr_diagram': (
        "an HR diagram (no spatial cone). "
        "Template: SELECT TOP <N> source_id, parallax, phot_g_mean_mag, bp_rp, ra, dec, pmra, pmdec, "
        "(phot_g_mean_mag + 5*LOG10(parallax/100)) AS abs_g_mag "
        "FROM gaiadr3.gaia_source "
        "WHERE parallax > 0 AND bp_rp IS NOT NULL "
        "ORDER BY random_index"
    ),
}

TARGETS = (
    'Pleiades (ra=56.75, dec=24.12), Orion (ra=83.8, dec=-5.4), '
    'Andromeda (ra=10.68, dec=41.27), LMC (ra=80.9, dec=-69.8), '
    'Galactic Centre (ra=266.4, dec=-29.0), Omega Centauri (ra=201.7, dec=-47.5), '
    "Barnard's Star (ra=269.45, dec=4.69)"
)

FILTER_HINTS = (
    'parallax > 5  (nearby), '
    'phot_g_mean_mag < 12  (bright), '
    'teff_gspphot > 5000  (hot), '
    'bp_rp > 1.5  (red)'
)

In [3]:
def build_prompt(intent: str, n: int) -> str:
    return f"""You are generating an evaluation dataset for a text-to-ADQL pipeline targeting the Gaia DR3 archive.

Generate exactly {n} (question, ADQL) pairs for the intent '{intent}': {INTENT_TEMPLATES[intent]}

STRICT RULES — every ADQL query MUST:
1. Use ONLY these columns: {', '.join(VALID_COLUMNS)}. Never invent column names.
2. Use the table gaiadr3.gaia_source (never anything else).
3. Follow the template above EXACTLY in structure, only varying the variables in <angle brackets>.
4. Use SELECT TOP N (N between 100 and 1000).
5. Have a WHERE clause and end with the ORDER BY clause shown in the template.
6. Contain at most ONE astrophysical filter on top of the template's mandatory clauses.
   Allowed simple filters: {FILTER_HINTS}.
   No JOINs, no BETWEEN, no AND-chains beyond the template, no subqueries, no aggregations.

For cone_search use real coordinates of well-known objects: {TARGETS}. Radius between 0.5 and 2.0 degrees.

Questions must be short, natural, and unambiguous — one clear ask, no compound requests, no mention of column names or the word 'intent'.
Write as a curious astronomy student would.

Return ONLY a JSON array, no markdown fences, no commentary:
[
  {{"question": "...", "sql": "SELECT TOP ...", "intent": "{intent}", "complexity": "Simple"}},
  ...
]
"""


def generate_for_intent(cursor, intent: str, n: int, model: str = 'mistral-large') -> list:
    prompt = build_prompt(intent, n)
    cursor.execute('SELECT SNOWFLAKE.CORTEX.COMPLETE(%s, %s)', (model, prompt))
    raw = cursor.fetchone()[0]
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    if not match:
        print(f'  {intent}: no JSON array found')
        return []
    try:
        return json.loads(match.group())
    except json.JSONDecodeError as e:
        print(f'  {intent}: JSON parse failed: {e}')
        return []


PER_INTENT = 10
MODEL      = 'mistral-large'

all_rows = []
for intent in INTENTS:
    print(f"Generating {PER_INTENT} pairs for '{intent}' ...")
    pairs = generate_for_intent(cursor, intent, PER_INTENT, MODEL)
    print(f'  -> got {len(pairs)} pairs')
    all_rows.extend(pairs)

df = pd.DataFrame(all_rows)
df.to_csv('gaia_eval_dataset.csv', index=False)

print(f'\nTotal: {len(df)} pairs saved to gaia_eval_dataset.csv')
print(df['intent'].value_counts())

Generating 10 pairs for 'cone_search' ...
  -> got 10 pairs
Generating 10 pairs for 'stellar_population' ...
  -> got 10 pairs
Generating 10 pairs for 'nearest_neighbours' ...
  -> got 10 pairs
Generating 10 pairs for 'hr_diagram' ...
  -> got 10 pairs

Total: 40 pairs saved to gaia_eval_dataset.csv
intent
cone_search           10
stellar_population    10
nearest_neighbours    10
hr_diagram            10
Name: count, dtype: int64


In [4]:
# Quick sanity check — show one example per intent
for intent in INTENTS:
    sub = df[df['intent'] == intent]
    if sub.empty:
        continue
    row = sub.iloc[0]
    print(f'\n[{intent}]')
    print(f'  Q: {row["question"]}')
    print(f'  SQL: {row["sql"]}')


[cone_search]
  Q: Find 500 bright stars around the Pleiades cluster.
  SQL: SELECT TOP 500 source_id, ra, dec, phot_g_mean_mag, bp_rp, pmra, pmdec FROM gaiadr3.gaia_source WHERE 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 56.75, 24.12, 1.0)) AND phot_g_mean_mag < 12 ORDER BY random_index

[stellar_population]
  Q: Which are the 500 hottest stars in the sky?
  SQL: SELECT TOP 500 source_id, ra, dec, teff_gspphot, phot_g_mean_mag, bp_rp, pmra, pmdec FROM gaiadr3.gaia_source WHERE teff_gspphot IS NOT NULL AND teff_gspphot > 5000 ORDER BY random_index

[nearest_neighbours]
  Q: What are the 500 closest red stars to the Sun?
  SQL: SELECT TOP 500 parallax, source_id, ra, dec, phot_g_mean_mag, bp_rp, pmra, pmdec FROM gaiadr3.gaia_source WHERE parallax > 0 AND bp_rp > 1.5 ORDER BY parallax DESC

[hr_diagram]
  Q: Show me an HR diagram of the 500 brightest stars.
  SQL: SELECT TOP 500 source_id, parallax, phot_g_mean_mag, bp_rp, ra, dec, pmra, pmdec, (phot_g_mean_mag + 5*LOG10(parallax